<h1 style="color:#004080; font-size:2em; margin-bottom:0.3em;">
  Build a HUC Lookup Table for STREAM Gages
</h1>
<h4 style="color:gray; font-weight:normal; line-height:1.5;">
  The STREAM Parquet resource on HydroShare doesn't include a HUC (Hydrologic Unit Code) field — the current metadata only provides <code>COMID</code>, the NHDPlus reach identifier for each gage's catchment. This notebook builds a small lookup table that maps each gage's <code>COMID</code> to its HUC6 and HUC8 codes, derived from the NHDPlus <code>REACHCODE</code> via the <code>pynhd</code> package. The result is cached locally so other notebooks (e.g. an advanced-queries notebook) can filter gages by HUC without re-querying the network every time.
</h4>

<div style="background-color: #f7f7f7; padding: 10px; border-radius: 5px;">
    <p><strong>Authors:</strong> 
    <ul>
        <li>Anthony Castronova, <a href="mailto:acastronova@cuahsi.org">acastronova@cuahsi.org</a></li>
        <li>Irene Garousi-Nejad, <a href="mailto:igarousi@cuahsi.org">igarousi@cuahsi.org</a></li>
    </ul>
    </p>
    <p><strong>Last Modified:</strong> 8/26/26</p>
    <p><strong>Supporting Files and Dependencies:</strong> 
        <ul>
            <li>pyproject.toml: adds <code>pynhd</code> and <code>netcdf4</code> (conda-forge, required to satisfy pynhd's transitive dependencies on Python 3.14)</li>
            <li>utils</li>
        </ul>
    </p>
    <p><strong>References:</strong></p>
    <ul>
        <li><a href="https://docs.hyriver.io/readme/pynhd.html">pynhd documentation (HyRiver suite)</a></li>
        <li>NHDPlus REACHCODE convention: the first 6/8 digits of a flowline's REACHCODE are its HUC6/HUC8</li>
    </ul>
</div>

<a id="authenticate"></a>
## Authenticate with HydroShare

In [1]:
from streamwq import S3hsclient

In [2]:
hs = S3hsclient.S3HydroShare()

Please Enter Your HydroShare Credentials


Load the STREAM metadata table, which contains each gage's `COMID`:

In [3]:
import pandas

resource_id = '4553ef8a14884e4199c7018975b8faaa'
resource = hs.resource(resource_id)

metadata_df = pandas.read_parquet(
    f'tonycastronova/{resource_id}/data/contents/metadata.parquet',
    filesystem=hs.get_s3_filesystem()
)

metadata_df[['STREAM_ID', 'COMID']].head()

,STREAM_ID,COMID
0,STREAM-gauge-1,18135842.0
1,STREAM-gauge-2,18135958.0
2,STREAM-gauge-3,18105412.0
3,STREAM-gauge-4,18122654.0
4,STREAM-gauge-5,18122646.0


## Query NHDPlus for REACHCODE by COMID

Each gage's `COMID` can be used to query NHDPlus flowline attributes via `pynhd`, which hits USGS/EPA web services directly — no download or authentication needed. We batch the COMIDs into chunks to keep each request a reasonable size, and show progress with `tqdm` since this queries several thousand gages over the network.

In [4]:
import pynhd
from tqdm import tqdm


def fetch_reachcodes(comids, batch_size=200):
    """Query NHDPlus for REACHCODE for a list of COMIDs, in batches."""
    wd = pynhd.WaterData('nhdflowline_network')
    results = []

    batches = [comids[i:i + batch_size] for i in range(0, len(comids), batch_size)]
    for batch in tqdm(batches, desc='Querying NHDPlus'):
        try:
            gdf = wd.byid('comid', batch)
            results.append(gdf[['comid', 'reachcode']])
        except Exception as e:
            print(f'Batch failed ({len(batch)} COMIDs): {e}')

    return pandas.concat(results, ignore_index=True)

Drop gages with a missing `COMID` (a handful of sites don't have a delineated catchment), then run the batched lookup:

In [5]:
comids = (
    metadata_df.dropna(subset=['COMID'])
    ['COMID']
    .astype('int64')
    .astype(str)
    .unique()
    .tolist()
)

print(f'{len(comids)} unique COMIDs to look up')

reachcodes_df = fetch_reachcodes(comids)
reachcodes_df

3828 unique COMIDs to look up


Querying NHDPlus: 100%|██████████| 20/20 [00:19<00:00,  1.03it/s]


,comid,reachcode
0,250456,11050002000088
1,369211,11110101000054
2,389904,11100301000048
3,391180,11100301000027
4,391658,11100301000024
...,...,...
3823,23752508,17090001000355
3824,23759334,17090002000046
3825,23773217,17090004000246
3826,23773229,17090004000251


## Derive HUC6 / HUC8 from REACHCODE

By NHDPlus convention, the first 6 digits of `REACHCODE` are the HUC6, and the first 8 are the HUC8 — no separate join table is needed.

In [6]:
reachcodes_df['HUC6'] = reachcodes_df['reachcode'].str[:6]
reachcodes_df['HUC8'] = reachcodes_df['reachcode'].str[:8]

reachcodes_df.head()

,comid,reachcode,HUC6,HUC8
0,250456,11050002000088,110500,11050002
1,369211,11110101000054,111101,11110101
2,389904,11100301000048,111003,11100301
3,391180,11100301000027,111003,11100301
4,391658,11100301000024,111003,11100301


## Merge back into the gage metadata and cache locally

Join this lookup table back to `metadata_df` on `COMID`, then save it locally so other notebooks can reuse it without re-querying NHDPlus:

In [7]:
import os

reachcodes_df = reachcodes_df.rename(columns={'comid': 'COMID'})
reachcodes_df['COMID'] = reachcodes_df['COMID'].astype('int64')

huc_lookup = metadata_df[['STREAM_ID', 'COMID']].merge(reachcodes_df, on='COMID', how='left')

os.makedirs('data', exist_ok=True)
output_path = 'data/gage_huc_lookup.parquet'
huc_lookup.to_parquet(output_path, index=False)

print(f'Saved {len(huc_lookup)} rows to {output_path}')
huc_lookup.head()

Saved 4268 rows to data/gage_huc_lookup.parquet


,STREAM_ID,COMID,reachcode,HUC6,HUC8
0,STREAM-gauge-1,18135842.0,03170005000618,031700,03170005
1,STREAM-gauge-2,18135958.0,03170005002640,031700,03170005
2,STREAM-gauge-3,18105412.0,03170007000066,031700,03170007
3,STREAM-gauge-4,18122654.0,03170006000851,031700,03170006
4,STREAM-gauge-5,18122646.0,03170006000005,031700,03170006


## Example: filter gages by HUC6

In [8]:
example_huc6 = huc_lookup['HUC6'].value_counts().idxmax()

subset = huc_lookup[huc_lookup['HUC6'] == example_huc6]
print(f'{len(subset)} gages in HUC6 {example_huc6}')
subset

259 gages in HUC6 170900


,STREAM_ID,COMID,reachcode,HUC6,HUC8
716,STREAM-gauge-848,23737983.0,17090012012210,170900,17090012
818,STREAM-gauge-952,23736397.0,17090012012177,170900,17090012
3504,STREAM-gauge-3992,23752118.0,17090001000172,170900,17090001
3505,STREAM-gauge-3993,23752030.0,17090001000128,170900,17090001
3506,STREAM-gauge-3994,23752002.0,17090001000114,170900,17090001
...,...,...,...,...,...
4245,STREAM-gauge-5057,23773229.0,17090004000251,170900,17090004
4250,STREAM-gauge-5064,23785767.0,17090006000076,170900,17090006
4261,STREAM-gauge-5081,23773217.0,17090004000246,170900,17090004
4266,STREAM-gauge-5086,23759334.0,17090002000046,170900,17090002


This cached lookup table (`data/gage_huc_lookup.parquet`) can now be loaded directly — e.g. `pandas.read_parquet('data/gage_huc_lookup.parquet')` — to filter gages by HUC6/HUC8 in other notebooks, such as the advanced-queries notebook, without needing to re-run the NHDPlus query.